Import Library

In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelBinarizer
from tensorflow.keras.utils import to_categorical
import cv2

Folder dan input dataset

In [2]:
data_dir = r"C:\Users\Lenovo\Klasifikasi_ikan\datasets\normalized_output"
classes = ["segar", "tidak_segar"]
img_height, img_width = 224, 224

#  Input Dataset 
print("📥 Memuat dataset...")
data, labels = [], []
for label in classes:
    path = os.path.join(data_dir, label)
    for img_name in os.listdir(path):
        img_path = os.path.join(path, img_name)
        img = cv2.imread(img_path)
        if img is not None:
            img = cv2.resize(img, (img_width, img_height))
            data.append(img)
            labels.append(label)

data = np.array(data, dtype="float32") / 255.0
labels = np.array(labels)
print(f"✅ Total data: {len(data)} gambar")

📥 Memuat dataset...
✅ Total data: 1000 gambar


Konfigurasi K-fold

In [3]:
# Encoding Label
lb = LabelBinarizer()
labels = lb.fit_transform(labels)
labels = to_categorical(labels)
num_classes = len(lb.classes_)

# Konfigurasi K-Fold 
k_folds = 3  # hanya 3 fold agar tuning lebih cepat
kf = KFold(n_splits=k_folds, shuffle=True, random_state=42)

Daftar tuning dan menyimpan hasil

In [4]:
learning_rates = [1e-4, 3e-4, 1e-3]
batch_sizes = [16, 32]
dropout_rates = [0.3, 0.4, 0.5]

results = []

Proses

In [5]:
for lr in learning_rates:
    for bs in batch_sizes:
        for dr in dropout_rates:
            acc_per_fold = []
            print(f"\n🚀 Uji kombinasi: LR={lr}, BS={bs}, Dropout={dr}")

            fold_no = 1
            for train_idx, val_idx in kf.split(data, labels):
                X_train, X_val = data[train_idx], data[val_idx]
                y_train, y_val = labels[train_idx], labels[val_idx]

                base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(img_height, img_width, 3))
                base_model.trainable = False

                x = base_model.output
                x = GlobalAveragePooling2D()(x)
                x = Dropout(dr)(x)
                predictions = Dense(num_classes, activation='softmax')(x)
                model = Model(inputs=base_model.input, outputs=predictions)

                model.compile(optimizer=Adam(learning_rate=lr),
                              loss='categorical_crossentropy',
                              metrics=['accuracy'])

                callbacks = [
                    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
                ]

                history = model.fit(
                    X_train, y_train,
                    validation_data=(X_val, y_val),
                    epochs=15,  # cukup pendek untuk tuning
                    batch_size=bs,
                    verbose=0,
                    callbacks=callbacks
                )

                scores = model.evaluate(X_val, y_val, verbose=0)
                acc_per_fold.append(scores[1] * 100)
                print(f"   Fold {fold_no}: {scores[1]*100:.2f}%")
                fold_no += 1

            avg_acc = np.mean(acc_per_fold)
            results.append((lr, bs, dr, avg_acc))
            print(f"📊 Rata-rata akurasi: {avg_acc:.2f}%")


🚀 Uji kombinasi: LR=0.0001, BS=16, Dropout=0.3




   Fold 1: 90.42%
   Fold 2: 90.99%
   Fold 3: 89.19%
📊 Rata-rata akurasi: 90.20%

🚀 Uji kombinasi: LR=0.0001, BS=16, Dropout=0.4
   Fold 1: 91.32%
   Fold 2: 91.59%
   Fold 3: 89.49%
📊 Rata-rata akurasi: 90.80%

🚀 Uji kombinasi: LR=0.0001, BS=16, Dropout=0.5
   Fold 1: 91.32%
   Fold 2: 90.09%
   Fold 3: 88.89%
📊 Rata-rata akurasi: 90.10%

🚀 Uji kombinasi: LR=0.0001, BS=32, Dropout=0.3
   Fold 1: 89.52%
   Fold 2: 87.99%
   Fold 3: 87.99%
📊 Rata-rata akurasi: 88.50%

🚀 Uji kombinasi: LR=0.0001, BS=32, Dropout=0.4
   Fold 1: 88.92%
   Fold 2: 88.29%
   Fold 3: 89.49%
📊 Rata-rata akurasi: 88.90%

🚀 Uji kombinasi: LR=0.0001, BS=32, Dropout=0.5
   Fold 1: 88.62%
   Fold 2: 88.29%
   Fold 3: 86.49%
📊 Rata-rata akurasi: 87.80%

🚀 Uji kombinasi: LR=0.0003, BS=16, Dropout=0.3
   Fold 1: 93.11%
   Fold 2: 94.89%
   Fold 3: 90.69%
📊 Rata-rata akurasi: 92.90%

🚀 Uji kombinasi: LR=0.0003, BS=16, Dropout=0.4
   Fold 1: 92.81%
   Fold 2: 92.19%
  

Hasil

In [6]:
results.sort(key=lambda x: x[3], reverse=True)
best = results[0]
print("\n==============================")
print("🏆 Kombinasi Hyperparameter Terbaik")
print("==============================")
print(f"Learning Rate : {best[0]}")
print(f"Batch Size    : {best[1]}")
print(f"Dropout Rate  : {best[2]}")
print(f"Rata-rata Akurasi: {best[3]:.2f}%")


🏆 Kombinasi Hyperparameter Terbaik
Learning Rate : 0.001
Batch Size    : 16
Dropout Rate  : 0.3
Rata-rata Akurasi: 94.50%


Coba Ulang

In [2]:
import os
import numpy as np
import cv2
from tensorflow.keras.utils import to_categorical

# === Folder Dataset ===
path_segar = r"C:\Users\Lenovo\Klasifikasi_ikan\datasets\normalized_output\segar"
path_tidak_segar = r"C:\Users\Lenovo\Klasifikasi_ikan\datasets\normalized_output\tidak_segar"

img_height, img_width = 224, 224

# === Fungsi untuk membaca dan resize citra ===
def load_images_from_folder(folder, label):
    images = []
    labels = []
    for filename in os.listdir(folder):
        img_path = os.path.join(folder, filename)
        img = cv2.imread(img_path)
        if img is not None:
            img = cv2.resize(img, (img_width, img_height))
            img = img / 255.0  # Normalisasi [0–1]
            images.append(img)
            labels.append(label)
    return images, labels

# === Muat semua citra ===
images_segar, labels_segar = load_images_from_folder(path_segar, 0)  # Label 0 = Segar
images_tdk_segar, labels_tdk_segar = load_images_from_folder(path_tidak_segar, 1)  # Label 1 = Tidak Segar

# === Gabungkan data ===
X_data = np.array(images_segar + images_tdk_segar, dtype="float32")
y_data = np.array(labels_segar + labels_tdk_segar)

# === One-hot encoding label ===
y_data = to_categorical(y_data, num_classes=2)

print(f"✅ Dataset siap digunakan")
print(f"Total data: {len(X_data)}")
print(f"Shape X_data: {X_data.shape}")
print(f"Shape y_data: {y_data.shape}")


✅ Dataset siap digunakan
Total data: 1000
Shape X_data: (1000, 224, 224, 3)
Shape y_data: (1000, 2)


In [3]:
# === 1. Import Library ===
import os
import numpy as np
import itertools
from sklearn.model_selection import KFold
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# === 2. Parameter Umum ===
img_height, img_width = 224, 224
num_classes = 2
k_folds = 5
epochs = 30

# === 3. Siapkan Folder Penyimpanan ===
base_dir = r"C:\Users\Lenovo\Klasifikasi_ikan\hasil_model"
tuning_dir = os.path.join(base_dir, "tuning")
os.makedirs(tuning_dir, exist_ok=True)

# === 4. Dataset (pastikan X_data dan y_data sudah tersedia dari blok sebelumnya) ===
# Misal: X_data.shape = (1000, 224, 224, 3), y_data.shape = (1000, 2)
kf = KFold(n_splits=k_folds, shuffle=True, random_state=42)

# === 5. Daftar Hyperparameter yang Akan Diuji ===
learning_rates = [1e-4, 5e-5]
dropout_rates = [0.3, 0.5]
batch_sizes = [16, 32]

# Simpan hasil semua kombinasi
tuning_results = []

# === 6. Loop Semua Kombinasi Hyperparameter ===
for lr, dropout, batch in itertools.product(learning_rates, dropout_rates, batch_sizes):
    print(f"\n🔹 Menguji kombinasi: LR={lr}, Dropout={dropout}, Batch={batch}")
    acc_per_fold = []
    loss_per_fold = []

    fold_no = 1
    for train_idx, val_idx in kf.split(X_data, y_data):
        print(f"\n🌀 Fold {fold_no}/{k_folds} sedang dilatih...")

        X_train, X_val = X_data[train_idx], X_data[val_idx]
        y_train, y_val = y_data[train_idx], y_data[val_idx]

        # === Model MobileNetV2 ===
        base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(img_height, img_width, 3))
        base_model.trainable = False

        x = base_model.output
        x = GlobalAveragePooling2D()(x)
        x = Dropout(dropout)(x)
        predictions = Dense(num_classes, activation='softmax')(x)
        model = Model(inputs=base_model.input, outputs=predictions)

        model.compile(
            optimizer=Adam(learning_rate=lr),
            loss='categorical_crossentropy',
            metrics=['accuracy']
        )

        # === Callback dan Penyimpanan Model ===
        model_folder = os.path.join(tuning_dir, f"LR_{lr}_DO_{dropout}_BS_{batch}")
        os.makedirs(model_folder, exist_ok=True)
        checkpoint_path = os.path.join(model_folder, f"fold_{fold_no}.keras")

        callbacks = [
            EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
            ModelCheckpoint(filepath=checkpoint_path, save_best_only=True, monitor='val_accuracy', mode='max')
        ]

        # === Training ===
        history = model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=epochs,
            batch_size=batch,
            callbacks=callbacks,
            verbose=1
        )

        # === Evaluasi ===
        scores = model.evaluate(X_val, y_val, verbose=0)
        print(f"✅ Fold {fold_no} - Akurasi: {scores[1]*100:.2f}% | Loss: {scores[0]:.4f}")

        acc_per_fold.append(scores[1] * 100)
        loss_per_fold.append(scores[0])

        fold_no += 1

    # === Simpan Rata-Rata Tiap Kombinasi ===
    avg_acc = np.mean(acc_per_fold)
    avg_loss = np.mean(loss_per_fold)
    tuning_results.append({
        "learning_rate": lr,
        "dropout": dropout,
        "batch_size": batch,
        "mean_accuracy": avg_acc,
        "mean_loss": avg_loss
    })

    print(f"\n📊 Rata-Rata untuk kombinasi ini:")
    print(f"    Akurasi: {avg_acc:.2f}% | Loss: {avg_loss:.4f}")
    print("-" * 50)

# === 7. Simpan Hasil Hyperparameter Tuning ===
import pandas as pd
tuning_results_df = pd.DataFrame(tuning_results)
csv_path = os.path.join(tuning_dir, "tuning_results.csv")
tuning_results_df.to_csv(csv_path, index=False)

# === 8. Menampilkan Hasil Akhir ===
print("\n==============================")
print("🏁 HASIL AKHIR HYPERPARAMETER TUNING")
print("==============================")
print(tuning_results_df.sort_values(by="mean_accuracy", ascending=False))
print(f"\n✅ Semua hasil tuning disimpan di folder: {tuning_dir}")



🔹 Menguji kombinasi: LR=0.0001, Dropout=0.3, Batch=16

🌀 Fold 1/5 sedang dilatih...


Epoch 1/30


50/50 [==============================] - 7s 111ms/step - loss: 0.7528 - accuracy: 0.5987 - val_loss: 0.5183 - val_accuracy: 0.7700
Epoch 2/30
50/50 [==============================] - 5s 92ms/step - loss: 0.5572 - accuracy: 0.7200 - val_loss: 0.4237 - val_accuracy: 0.8450
Epoch 3/30
50/50 [==============================] - 5s 94ms/step - loss: 0.4854 - accuracy: 0.7937 - val_loss: 0.3718 - val_accuracy: 0.8800
Epoch 4/30
50/50 [==============================] - 4s 89ms/step - loss: 0.4740 - accuracy: 0.7850 - val_loss: 0.3224 - val_accuracy: 0.9150
Epoch 5/30
50/50 [==============================] - 4s 85ms/step - loss: 0.4132 - accuracy: 0.8400 - val_loss: 0.2948 - val_accuracy: 0.9150
Epoch 6/30
50/50 [==============================] - 4s 88ms/step - loss: 0.4119 - accuracy: 0.8213 - val_loss: 0.2754 - val_accuracy: 0.9150
Epoch 7/30
50/50 [==============================] - 4s 89ms/step